### Use this notebook to convert your worker1.log to a csv with features needed to build the Random Forest model

In [8]:
import pandas as pd 
import numpy as np
import rf_feature_pipeline as s4fp
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [9]:
def flatten_sparse_json(json_list, nested_key="fields"):
    """
    Creates a Pandas DataFrame by flattening all top-level keys and
    all keys within a specified nested dictionary (the 'fields' key).
    Missing values are automatically set to NaN.
    """
    
    # Use pd.json_normalize on the entire list.
    # By default, it automatically flattens nested dictionaries and prefixes
    # the new columns with the name of the nested key, followed by a dot.
    # E.g., 'fields': {'r': 3} becomes the column 'fields.r'.
    df = pd.json_normalize(json_list)

    # 2.1. Clean up Column Names
    # We rename the flattened columns by removing the 'fields.' prefix.
    # This loop only affects columns that start with 'fields.'.
    df = df.rename(columns=lambda x: x.replace(f'{nested_key}.', '') 
                                     if x.startswith(f'{nested_key}.') else x)
    
    # 2.2. Drop the original nested column
    # The original 'fields' column itself contains the dict object and is now redundant.
    if nested_key in df.columns:
         df = df.drop(columns=[nested_key])

    return df 

def read_log_as_csv(path_to_logfile:str):
    with open(path_to_logfile) as f:
        lines = f.readlines()
        log_in_json = []
        for line in lines:
            json_obj = json.loads(line)
            log_in_json.append(json_obj)
    return flatten_sparse_json(log_in_json, nested_key="fields")   

In [6]:
from pathlib import Path

workerlog_landlord_paths = []
for path in Path("/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/runs/").rglob("worker1.log"):
    a = str(path)
    if "landlord" in a.lower() and "precleanup" not in a.lower():
        workerlog_landlord_paths.append(a)

In [14]:
workerlog_landlord_paths[36:44]

['/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/runs/30_min/4.0/6.0/az-3/0/Landlord/2/12/worker1.log',
 '/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/runs/30_min/4.0/6.0/az-4/0/Landlord/2/4/worker1.log',
 '/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/runs/30_min/4.0/6.0/az-4/0/Landlord/2/12/worker1.log',
 '/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/runs/30_min/4.0/6.0/az-2/0/Landlord/2/4/worker1.log',
 '/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/runs/30_min/4.0/6.0/az-2/0/Landlord/2/12/worker1.log',
 '/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/runs/30_min/4.0/6.0/dynamic-4/0/Landlord/2/12/worker1.log',
 '/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/runs/30_min/4.0/6.0/dynamic-3/0/Landlord/2/12/worker1.log',
 '/Users/akshaykishan/PycharmProjects/iluvatar-faas/S4forIluvatar/src/data/runs/30

In [ ]:
main_df = pd.DataFrame()
for i, path in enumerate(workerlog_landlord_paths):
    raw_data = read_log_as_csv(path)
    data = s4fp.generate_rf_features(raw_data, lags=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 20])
    data['session'] = int(i)
    main_df = pd.concat([main_df, data], ignore_index=True)
    print(f"path {path} completed!")
    print("=="*40)

Delegating baseline feature extraction to S4 Pipeline...
Extracting detailed queue state history... this may take a moment.
Enhancing features with local temporal RF lags...
Generating rolling lag features across lags [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 20]...
RF Feature extraction complete!
path /Users/akshaykishan/Akshay/Research/Jan_Run_2/30_min/4.0/6.0/small-3/0/Landlord/2/4/worker1.log completed!
Delegating baseline feature extraction to S4 Pipeline...
Extracting detailed queue state history... this may take a moment.
Enhancing features with local temporal RF lags...
Generating rolling lag features across lags [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 20]...
RF Feature extraction complete!
path /Users/akshaykishan/Akshay/Research/Jan_Run_2/30_min/4.0/6.0/small-3/0/Landlord/2/12/worker1.log completed!
Delegating baseline feature extraction to S4 Pipeline...
Extracting detailed queue state history... this may take a moment.
Enhancing features with local temporal RF lags...

In [6]:
main_df.groupby('session')['invocation_timestamp'].count().max()

3052

In [7]:
[x for x in main_df.columns if 'lag' in x]

['target_queue_len_lag_1',
 'others_len_queue_lag_1',
 'num_running_funcs_filled_lag_1',
 'target_queue_len_lag_2',
 'others_len_queue_lag_2',
 'num_running_funcs_filled_lag_2',
 'target_queue_len_lag_3',
 'others_len_queue_lag_3',
 'num_running_funcs_filled_lag_3',
 'target_queue_len_lag_4',
 'others_len_queue_lag_4',
 'num_running_funcs_filled_lag_4',
 'target_queue_len_lag_5',
 'others_len_queue_lag_5',
 'num_running_funcs_filled_lag_5',
 'target_queue_len_lag_6',
 'others_len_queue_lag_6',
 'num_running_funcs_filled_lag_6',
 'target_queue_len_lag_7',
 'others_len_queue_lag_7',
 'num_running_funcs_filled_lag_7',
 'target_queue_len_lag_8',
 'others_len_queue_lag_8',
 'num_running_funcs_filled_lag_8',
 'target_queue_len_lag_9',
 'others_len_queue_lag_9',
 'num_running_funcs_filled_lag_9',
 'target_queue_len_lag_10',
 'others_len_queue_lag_10',
 'num_running_funcs_filled_lag_10',
 'target_queue_len_lag_11',
 'others_len_queue_lag_11',
 'num_running_funcs_filled_lag_11',
 'target_queue_

In [1]:
# main_df.to_csv("landloard_data_for_rf_training_w_lagged_features_20_coldstart.csv")

In [7]:
main_df.groupby('session')['invocation_timestamp'].count().max()

3052